In [1]:
# getting data from kaggle
import kagglehub

# Download latest version
# path = kagglehub.dataset_download("arashnic/book-recommendation-dataset", output_dir="./data")
path = kagglehub.dataset_download("arashnic/book-recommendation-dataset", path="Books.csv", output_dir="./data")

print("Path to dataset files:", path)

/home/tushar/Desktop/blank/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: ./data/Books.csv


In [109]:
import pandas as pd
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk


In [110]:
df = pd.read_csv("./data/Books.csv")[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher']].iloc[:20000]

/tmp/ipykernel_59944/2498094134.py:1: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("./data/Books.csv")[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher']].iloc[:20000]


In [111]:
df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company


In [112]:
es = Elasticsearch("http://localhost:9200")
target_index = "book_recommender"


In [113]:
es.indices.delete(index=target_index, ignore_unavailable=True)


ObjectApiResponse({'acknowledged': True})

In [114]:
mapping = {
    "mappings": {
        "properties": {
            "ISBN": {"type": "keyword"},
            "Book-Title": {"type": "text"},
            "Book-Author": {"type": "text"},
            "Year-Of-Publication": {"type": "integer"},
            "Publisher": {"type": "text"},
            "title_vector": {"type": "dense_vector",
                "dims": 384,
                "index": True,
                "similarity": "cosine"}
        }
    }
}

es.indices.create(index=target_index, body=mapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'book_recommender'})

In [115]:
# setup the embedding model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7236.77it/s]


In [116]:

# generator to stream DataFrame rows into Elasticsearch docs
def doc_generator(dataframe, index_name):
    for index, row in dataframe.iterrows():
        # convert row into standart python dictionary
        doc = row.to_dict()
        doc["title_vector"] = model.encode(doc["Book-Title"]).tolist()

        # structure the payload format required by the bulk API
        yield {
            "_index":index_name,
            "_id":doc.get("ISBN"),
            "_source":doc
        }

In [117]:
# Trigger the bulk ingestion process
try:
    success, failed = bulk(es, doc_generator(df, target_index), raise_on_error=False, stats_only=False)
    print(f"Successfully indexed: {success}")
    if failed:
        print(f"Failed to indexed docs: {failed}")
except Exception as e:
    print(f"An error occurred during indexing: {e}")

Successfully indexed: 20000


In [118]:
# Aside: pretty printing Elasticsearch responses

def pretty_response(response):
    if len(response['hits']['hits']) == 0:
        print("Your search returned no results.")
    else:
        for hit in response['hits']['hits']:
            id = hit["_id"]
            publication_date = hit["_source"]["Year-Of-Publication"]
            score = hit["_score"]
            title = hit["_source"]["Book-Title"]
            publisher = hit["_source"]["Publisher"]
            authors = hit["_source"]["Book-Author"]
            pretty_output = f"\nID: {id}\nPublication date: {publication_date}\nTitle: {title}\nPublisher: {publisher}\nAuthors: {authors}\nScore: {score}"
            print(pretty_output) 

In [124]:
response = es.search(
    index=target_index,
    knn={
        "field": "title_vector",
        "query_vector": model.encode("dog").tolist(),
        "k": 10,
        "num_candidates": 100,
    },
)

pretty_response(response)


ID: 0684835525
Publication date: 1997
Title: DOG LOVE
Publisher: Simon &amp; Schuster
Authors: Marjorie Garber
Score: 0.8614902

ID: 0743220633
Publication date: 2002
Title: Sun Dog
Publisher: Simon &amp; Schuster (Trade Division)
Authors: Monique Roffey
Score: 0.84513396

ID: 0679762671
Publication date: 1996
Title: A Dog's Life
Publisher: Vintage Books USA
Authors: Peter Mayle
Score: 0.8316518

ID: 067086935X
Publication date: 1996
Title: Dog Brain
Publisher: Viking Books
Authors: David Milgrim
Score: 0.81854904

ID: 0385494327
Publication date: 1999
Title: Black Dogs
Publisher: Anchor Books/Doubleday
Authors: Ian McEwan
Score: 0.81164014

ID: 0340818689
Publication date: 2001
Title: The dog catcher
Publisher: Sceptre
Authors: Alexei Sayle
Score: 0.8111142

ID: 0553284118
Publication date: 1990
Title: Creature
Publisher: Bantam Books
Authors: John Saul
Score: 0.80758405

ID: 0843922567
Publication date: 1985
Title: Creature
Publisher: Banner of Truth
Authors: Drake Douglas
Score: 0.